## What is MLflow and what does it solve?

As soon as you train more than a handful of ML models, the usual ad-hoc tooling — a notebook with the latest cell in scroll-back, a `results.xlsx` of metrics, a folder of `model_v3_final_FINAL.pkl` pickles, a Slack DM asking a colleague which hyperparameters they used — stops scaling. You lose track of which run produced which metric, which code commit trained which model, and which artifact is safe to deploy.

**MLflow** is an open-source platform that replaces that improvisation with four components, usable à la carte:

| Component | What it does | Replaces |
|---|---|---|
| **Tracking** | Logs parameters, metrics, tags, and artifacts (plots, model files, datasets) for every training run, keyed by a unique run ID. | Spreadsheets and filename conventions for "which numbers came from which run". |
| **Models** | A standard packaging format (`pyfunc`) so a model trained in any framework can be loaded back uniformly — in Python, REST, batch, or Spark. | Re-implementing inference code per deployment target. |
| **Model Registry** | A versioned catalog of the models you actually care about, with stage labels (`Staging`, `Production`, …) and an audit trail of who promoted what when. | "Ask the author which pickle is the good one." |
| **Deployment / Serving** | Turns a registered model into a REST endpoint or batch inference job through the same `pyfunc` interface. | Hand-rolled Flask wrappers around a pickle. |

In ML-system terms, MLflow is the layer between *"I trained a model in a notebook"* and *"we serve a specific, traceable version of it in production"* — it provides the reproducibility, observability, collaboration, and governance that distinguish production ML from a one-off experiment. The rest of this repo's tutorials walk through these components in order; **this notebook sets up the local infrastructure — a tracking server and a database — that all of them assume is running.**

# Connect Your Development Environment to MLflow

This guide shows you how to connect your Machine Learning development workflow to MLflow. You can run MLflow:

- **Locally**, with the client and server on the same machine (this notebook).
- **Self-hosted**, on a shared server somewhere on your network or in the cloud.
- **Managed**, e.g. [Databricks Managed MLflow](https://mlflow.org/docs/latest/getting-started/databricks-trial.html).

This notebook focuses on the **local OSS MLflow** path, which is what the rest of the tutorials in this repo assume. It's a slight modification of the [official setup guide](https://mlflow.org/docs/latest/ml/getting-started/running-notebooks/). Anywhere this notebook corrects, supplements, or diverges from upstream, the relationship is named with an inline bold callout — one of **Bug**, **Stale**, **Missing from**, or **Diverges from upstream tutorial:** — or, in code cells, with a `NOTE (differs from upstream)` comment. Sections with no upstream counterpart use a plain heading and no callout.

## Prerequisites

Two requirements before any cell in this notebook will work:

- **Python 3.14+** — pinned by `requires-python = ">=3.14"` in `pyproject.toml`. If you're on an older Python, `uv sync` will refuse to resolve and the notebook kernel won't start.
- **The repo's virtual environment activated.** Without it, `import mlflow` either fails (no system-wide install) or imports an unrelated MLflow version from another project — leaving you to debug mysterious version mismatches. With `direnv` installed, `.envrc` activates the venv automatically when you `cd` into the repo; otherwise run `source .venv/bin/activate` from the repo root.

## Step 1: Install MLflow

**Diverges from upstream tutorial:** Upstream says `pip install --upgrade "mlflow>=3.1"`. This repo uses **`uv` exclusively** for dependency management — `pip` is forbidden, including `uv pip install` (see `CLAUDE.md`). MLflow is already in `pyproject.toml`, so a fresh checkout only needs:

```bash
uv sync
```

Adding or upgrading a dependency goes through the project workflow so `pyproject.toml` and `uv.lock` stay the single source of truth:

```bash
uv add 'mlflow>=3.1'             # add or pin
uv add --upgrade-package mlflow  # upgrade an already-pinned package
```

**Databricks variant:** if you ever connect this notebook to Databricks Managed MLflow, install the `[databricks]` extra: `uv add 'mlflow[databricks]>=3.1'`. For the local-only path covered here it isn't needed.

## Step 2: Configure tracking

Every MLflow client call needs to know *where* to log to. That "where" is the **tracking URI**.

> **What's a URI?** A **URI** (Uniform Resource Identifier) is a string that names a resource and how to reach it. The **scheme prefix** says *how*: `http://` means "over HTTP to this host:port", `sqlite:///` means "a SQLite database file at this path", `file:` means "a plain directory on disk". You'll see all three below. (URLs — `http://`-style web addresses — are a subset of URIs; for our purposes the terms are interchangeable.)

There are three common shapes:

| Option | Tracking URI on the client | What runs locally |
|---|---|---|
| **A — Database (recommended for local dev)** | `sqlite:///mlflow.db` | Nothing extra; the client writes directly to the SQLite file. |
| **B — File system** *(legacy, KTLO mode)* | `file:./mlruns` or unset | Nothing extra; the client writes flat files into `./mlruns/`. |
| **C — Remote tracking server** | `http://<host>:<port>` | A running `mlflow server` process the client talks to over HTTP. |

The rest of the tutorials in this repo use **Option C** — they expect a tracking server running on `http://127.0.0.1:5000`. We'll set that up after the next two sections.

> **A note on Option B (file-system backend):** the upstream docs label this **"TO BE DEPRECATED SOON"** — it doesn't support concurrent writes and, critically, **does not support the Model Registry**. Calls like `mlflow.register_model(...)` or `registered_model_name=...` simply fail against a file backend. Treat Option B as a curio; use Option A or C for anything you care about.

## `mlflow ui` vs `mlflow server`

Reading MLflow docs, Stack Overflow answers, and AI-generated summaries, you'll see both `mlflow ui` and `mlflow server` used to start the local tracking server, and many sources describe them as different commands. A typical (and **incorrect** for MLflow 3.x) framing looks like:

> *"`mlflow ui` is for local, quick development on a single machine, while `mlflow server` is geared toward production, allowing remote access, artifact storage, and handling multiple users."*

**In MLflow 3.x, that distinction no longer exists. `mlflow ui` is a string-level alias for `mlflow server`** — same defaults (`--host 127.0.0.1`, `--port 5000`, `--workers 4`), same flags, same Gunicorn-backed server. Anything you can do with one you can do with the other by changing flags.

### Evidence 1: the CLI helps are byte-identical

```bash
$ diff <(mlflow ui --help) <(mlflow server --help)
1c1
< Usage: mlflow ui [OPTIONS]
---
> Usage: mlflow server [OPTIONS]
```

Only the program name on line 1 differs. Every flag, every default, every help paragraph is the same.

### Evidence 2: the source code makes the equivalence explicit

From `mlflow/cli/__init__.py` in mlflow `3.12.0` (lines 63–67):

```python
class AliasedGroup(click.Group):
    def get_command(self, ctx, cmd_name):
        # `mlflow ui` is an alias for `mlflow server`
        cmd_name = "server" if cmd_name == "ui" else cmd_name
        return super().get_command(ctx, cmd_name)
```

Click rewrites `ui` → `server` *before* dispatch. There is one implementation; the `ui` name never reaches a different code path. So claims like "`mlflow ui` doesn't support Postgres", "`mlflow ui` only runs one worker", or "`mlflow ui` binds to 127.0.0.1 *unlike* `mlflow server`" are all false on MLflow 3.x — both commands accept the same `--backend-store-uri`, the same `--workers`, and both default to `--host 127.0.0.1`.

### So when do people use which name?

Purely convention now:

- `mlflow ui` reads as "give me the local UI for my experiments" — common in quickstart docs and tutorials.
- `mlflow server` reads as "start the tracking service" — common in deployment scripts, systemd units, Docker entrypoints, and team-shared setups.

Pick whichever name fits the *intent* the reader will infer; under the hood you get the same process either way. This repo's notebooks use `mlflow server` in setup instructions because we're treating the local tracking server as a "service" the notebooks talk to over HTTP.

### Historical context (pre-MLflow 3)

The "ui = quick local, server = production" framing was **accurate at one point**, which is why it's still circulating. In older MLflow versions:

- `mlflow ui` ran a single-process Flask dev server, defaulted to the file backend (`./mlruns`), and didn't expose `--workers` or many of the production flags.
- `mlflow server` ran under Gunicorn with multiple workers, accepted `--workers` / `--gunicorn-opts`, and was the documented entry point for multi-user deployments.

If you find a blog post or AI answer that describes the two as different, check its date (or its training cutoff). The unification happened in MLflow 3.

## Client tracking URI vs server backend store URI

The single most common source of "my run vanished!" confusion in MLflow is mixing up two different URIs that both sound like "where MLflow stores things":

| URI | Lives on | Set with | What it answers |
|---|---|---|---|
| **Tracking URI** | The **client** (your notebook / training script) | `mlflow.set_tracking_uri(...)` or `MLFLOW_TRACKING_URI` env var | *Where do I send my log_param / log_metric / log_model calls?* |
| **Backend store URI** | The **server** (`mlflow server` process) | `--backend-store-uri` flag | *Where does the server persist what it receives?* |

These are independent. A few illustrative combinations:

**Local-only, Option A (no server, direct SQLite):**
```python
mlflow.set_tracking_uri("sqlite:///mlflow.db")  # client writes directly to sqlite
```
There is no server. The client *is* the backend.

**Local-only, Option C (server in front of SQLite):**
```bash
mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000
```
```python
mlflow.set_tracking_uri("http://127.0.0.1:5000")  # client talks to server over HTTP
```
The server persists to SQLite; the client doesn't know or care what backend the server chose.

> **What's a "foot-gun"?** Programmer slang, from *"a gun aimed at your own foot"*: a feature or default that is easy to use the wrong way and silently produce a result that isn't what you wanted. No error is raised, the call succeeds, the code looks correct — but the outcome is subtly broken. The next example is exactly that.

**Mismatched defaults — the classic foot-gun:**
```bash
# Terminal 1
mlflow server --port 5000
```
```python
# Notebook
mlflow.set_tracking_uri("sqlite:///mlflow.db")  # NOT the server!
```
The notebook writes runs directly into `mlflow.db`. The server is also using `mlflow.db` by default (in MLflow 3), so if both processes run from the **same** working directory you'll *see* the runs in the UI — but they got there bypassing the HTTP layer, which means any server-side feature (auth, artifact proxy, audit logging) is silently skipped.

**The correct rule: when running a server, point the client at `http://...`, not at the same database. Anything else is a coincidence-driven setup that breaks the moment you change the cwd, add auth, or move to a remote backend.**

### Demonstrating the foot-gun

The difference between the correct setup and the foot-gun is **invisible in the API** — both calls succeed silently — but **observable on disk and in the UI**. Assume the server is already running at `http://127.0.0.1:5000` from the repo root, and you run the snippets below from a notebook whose `cwd` is `src/basics/`.

**A. Correct setup — client points at the server**

```python
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("foot-gun-demo-correct")

with mlflow.start_run():
    mlflow.log_param("alpha", 0.1)

print(mlflow.get_tracking_uri())   # http://127.0.0.1:5000
```

Observable result:
- The run shows up at <http://127.0.0.1:5000/> under the `foot-gun-demo-correct` experiment.
- **No** new files appear in `src/basics/`.
- The server's `mlflow.db` at the repo root gains a row in `runs`, and the artifact (none in this case — no `log_model`) would land under the repo-root `mlartifacts/`.

**B. Foot-gun — client bypasses the server**

```python
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")   # relative path → resolved against cwd
mlflow.set_experiment("foot-gun-demo-bypass")

with mlflow.start_run():
    mlflow.log_param("alpha", 0.1)

print(mlflow.get_tracking_uri())   # sqlite:///mlflow.db
```

Observable result:
- The run does **not** appear in the server's UI.
- A **second** `mlflow.db` is created at `src/basics/mlflow.db` — a separate database, distinct from the server's `mlflow.db` at the repo root. Same name, different file.
- The server has no knowledge of this run. Any server-side feature — auth, artifact proxy, audit logging, the model registry — is silently skipped.

**Quick diagnostic, runnable from any cell:**

```python
mlflow.get_tracking_uri()
```

- Returns `http://...` → client is talking to the server (correct setup).
- Returns `sqlite:///...` or `file:./mlruns` → client is writing directly to its cwd. That is intentional for Option A; it is the foot-gun for Option C.

**Recovery if you accidentally ran the foot-gun:** the rogue `mlflow.db` and `mlartifacts/` directories under `src/basics/` are pure per-developer state — safe to `rm -rf`. After deleting them, reset the tracking URI to `http://127.0.0.1:5000` and re-run.

## What the server's default backend really is

**Stale in upstream tutorial:** The `mlflow server --backend-store-uri` help text says:

> By default, data will be logged to the `./mlruns` directory.

Since [PR #18497](https://github.com/mlflow/mlflow/pull/18497) (MLflow 3), the actual default is:

- If `./mlruns/` already exists in the cwd → keep using the file backend (`file:./mlruns`) for backward compatibility.
- Otherwise → create and use **`sqlite:///mlflow.db`** in the cwd.

**Why this matters:** when you start `mlflow server --port 5000` from this repo's root (no existing `mlruns/`), you automatically get a SQLite-backed server that supports the Model Registry, transactions, and everything else the file backend can't do — **with no flags required**. The whole point of Option A's `--backend-store-uri sqlite:///mlflow.db` argument in the upstream tutorial is now the default.

## Step 3: Start the tracking server (Option C)

Open a separate terminal at the **repo root** (so `mlflow.db` and `mlartifacts/` land predictably) and run:

```bash
mlflow server --host 127.0.0.1 --port 5000
```

You should see startup logs like:

```text
Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only).
```

Now point this notebook at it and create (or re-attach to) an experiment.

In [ ]:
import mlflow

HOST = "127.0.0.1"
PORT = 5000
mlflow.set_tracking_uri(uri=f"http://{HOST}:{PORT}")

mlflow.set_experiment("my-first-experiment")

## Step 4: Verify the connection

The next cell is a full health-check, not just a one-shot log. It answers four questions in order:

1. **Where is the client sending calls?** → `mlflow.get_tracking_uri()`.
2. **Where will registered models go?** → `mlflow.get_registry_uri()`. In MLflow 3 the registry URI defaults to the tracking URI, so these usually match; they only differ if you explicitly call `mlflow.set_registry_uri(...)` or set `MLFLOW_REGISTRY_URI`.
3. **Is the server actually up, and which MLflow version is it running?** → a plain HTTP `GET /version` against the tracking URI. This is the same endpoint the UI uses; it confirms the HTTP layer is alive.
4. **Is the persistence layer behind the server responding?** → `mlflow.search_experiments()`. A successful list (even an empty one) means the server can reach its backend store; a hung or failing call points at the database, not the HTTP port.

Only after those four does the cell actually log a parameter — the canonical "I wrote something and got a run_id back" smoke test. If the server is *not* up, the cell raises a connection error pointing at port 5000 — that's the obvious failure mode to recognize.

In [5]:
import urllib.request

# 1. Where is the client sending calls?
print(f"Tracking URI : {mlflow.get_tracking_uri()}")

# 2. Where will registered models go? (defaults to tracking URI in MLflow 3)
print(f"Registry URI : {mlflow.get_registry_uri()}")

# 3. Is the server up? Ask its /version endpoint directly.
with urllib.request.urlopen(f"http://{HOST}:{PORT}/version") as resp:
    server_version = resp.read().decode().strip()
print(f"Server reports MLflow version: {server_version}")

# 4. Can the server reach its backend store? (lists experiments, including 'Default')
experiments = mlflow.search_experiments()
print(f"\nServer knows about {len(experiments)} experiment(s):")
for exp in experiments:
    print(f"  - {exp.name:<30} (id={exp.experiment_id})")

# 5. Actually write something — the canonical end-to-end smoke test.
print()
with mlflow.start_run(run_name="connection_check") as run:
    mlflow.log_param("test_param", "test_value")
    print(f"Logged run_id: {run.info.run_id}")

print("\nSuccessfully connected to MLflow!")

Tracking URI : http://127.0.0.1:5001
Registry URI : http://127.0.0.1:5001
Server reports MLflow version: 3.12.0

Server knows about 2 experiment(s):
  - my-first-experiment            (id=1)
  - Default                        (id=0)

Logged run_id: 996ade8d1e7444f6a2469c78f2f1b59d
🏃 View run connection_check at: http://127.0.0.1:5001/#/experiments/1/runs/996ade8d1e7444f6a2469c78f2f1b59d
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1

Successfully connected to MLflow!


### What about the server's backend store?

Notice none of the calls above told us *where the server is actually persisting our runs* — `sqlite:///mlflow.db`? Postgres? Something else? That omission is **intentional**. The backend store URI is server-side configuration and is deliberately not exposed over the REST API. The client shouldn't need to know — that's the whole point of running a tracking server in front of the database: auth, audit logging, schema migrations, swapping SQLite for Postgres later, are all transparent to client code.

For local development you can still see what the server chose, by looking at things you control directly:

1. **The server's startup log.** Scroll back in the terminal where you ran `mlflow server`. Near the top you'll find:
   ```text
   Backend store URI not provided. Using sqlite:///mlflow.db
   Registry store URI not provided. Using backend store URI.
   ```
   That is the authoritative answer.

2. **The working directory the server was started from.** Default paths (`mlflow.db`, `mlartifacts/`) are *relative to the server's cwd*. From the repo root:
   ```bash
   ls -lh src/mlflow.db src/mlartifacts/
   ```
   Both should exist after the cell above ran successfully.

3. **(Optional) Peek inside the SQLite file.** Same directory:
   ```bash
   sqlite3 mlflow.db ".tables"
   sqlite3 mlflow.db "SELECT name FROM experiments;"
   ```
   You'll see tables like `experiments`, `runs`, `metrics`, `params`, `registered_models`, and rows for `Default` and `my-first-experiment`.

If you ever lose track of what the server is configured with — for example because you started it days ago in a background tmux pane — restarting it is the fastest answer: the `Backend store URI ...` log line is printed on every boot.

## Step 5: Access the MLflow UI

Browse to <http://127.0.0.1:5000/> in your browser (same host and port the `mlflow server` process is listening on). You should see:

- An **Experiments** sidebar with `Default` and `my-first-experiment`.
- One run named `connection_check` under `my-first-experiment`, with the `test_param = test_value` parameter logged.

### "Invalid Host header" when accessing from another machine

If you started the server with `--host 0.0.0.0` to expose it on your LAN and the UI returns:

> Invalid Host header — possible DNS rebinding attack detected

that's MLflow 3's built-in security middleware refusing requests with a non-localhost `Host:` header. The fix is to allowlist the hostnames you'll connect from when you start the server:

```bash
mlflow server \
    --host 0.0.0.0 \
    --port 5000 \
    --allowed-hosts 127.0.0.1,my-laptop.local \
    --cors-allowed-origins http://my-laptop.local:5000
```

For purely local development, `--host 127.0.0.1` (the default) avoids the issue entirely.

### The "Traces" tab — orientation (so the empty menu doesn't puzzle you)

In the MLflow 3 UI you'll have noticed a **Traces** tab next to Runs and Models — and for the **traditional-ML** notebooks (the `ml/` track) it's **empty**. That's expected, not a misconfiguration:

- **What it is.** MLflow *Tracing* records OpenTelemetry-style **spans** (inputs/outputs/latency for each step) — built for **GenAI / LLM apps and agents** (LangChain, OpenAI, LlamaIndex, …), where you need to see *inside* a chain or agent call.
- **Why it's empty for the ML track.** Traditional-ML runs (sklearn/XGBoost `fit`/`predict`) emit no spans.
- **Where it lights up.** The **`gen_ai/` track** — instrumenting LLM code with `@mlflow.trace` or a GenAI autolog flavor is what fills this tab.

This note exists for **MLflow-3 UI literacy**: now you know what the tab is, why the ML track doesn't fill it, and which track does.

## What's next

Now that the server is up and the client can reach it, continue with the shared tracking quickstart, then branch into whichever track fits your work:

- **`b_tracking_quickstart.ipynb`** (next, here in `basics/`) — the 5-minute tour of tracking: experiments, runs, logging params / metrics / a model, and loading it back as a pyfunc.

Then pick a track:

- **`../ml/` — traditional ML** (sklearn / XGBoost): hyperparameter tuning, plots, evaluation, the model registry, and serving. Start with `ml/b_hyperparameter_tuning.ipynb`.
- **`../gen_ai/` — GenAI / LLM apps**: tracing, LLM-as-judge evaluation, and the prompt registry — this is where the **Traces** tab above lights up. *(In progress — see `roadmap/`.)*

### Other connection paths (out of scope for this repo)

- [Databricks Managed MLflow](https://mlflow.org/docs/latest/getting-started/databricks-trial.html) — set `MLFLOW_TRACKING_URI=databricks` plus `DATABRICKS_HOST` / `DATABRICKS_TOKEN`. The upstream tutorial covers this in its "Databricks - Local IDE" tab.
- [Self-hosting a shared tracking server](https://mlflow.org/docs/latest/self-hosting.md) — put `mlflow server` behind a reverse proxy with auth, point a Postgres backend at it, and share with your team.